In [1]:
import pandas as pd
import numpy as np

In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import pickle, os

In [2]:
df = pd.read_csv("merged_disease_symptoms_data.csv")

In [3]:

# Handle missing values
df["Type of Disease"] = df["Type of Disease"].fillna("Unknown")
df["Weight"] = df["Weight"].fillna(2)
df["Occurrence"] = df["Occurrence"].fillna("Common")

# Replace rare diseases (<10 samples) with "Other"
counts = df["diseases"].value_counts()
rare = counts[counts < 10].index
df["diseases"] = df["diseases"].replace(rare, "Other")

In [5]:
dataset_info = df.info()
dataset_head = df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 246945 entries, 0 to 246944
Columns: 381 entries, diseases to Weight
dtypes: float64(1), int64(377), object(3)
memory usage: 717.8+ MB


In [10]:
# Select symptom columns (binary encoded)
symptom_cols = [
    col for col in df.columns if col not in ["diseases", "Type of Disease", "Weight", "Occurrence"]
]
df[symptom_cols] = df[symptom_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

In [11]:
# 🔀 Convert to Multi-Label Format
# ================================================
# Group rows that share the same symptom pattern
grouped = df.groupby(symptom_cols).agg({
    "diseases": lambda x: list(set(x)),
    "Weight": "mean"
}).reset_index()

# Features (symptoms)
X = grouped[symptom_cols].values

# Multi-label target encoding
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(grouped["diseases"])

num_classes = len(mlb.classes_)
input_dim = X.shape[1]

print(f"✅ Dataset ready: {X.shape[0]} samples, {X.shape[1]} symptoms, {num_classes} unique diseases.")


C:\Users\ASD\AppData\Local\Temp\ipykernel_1132\3687761658.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()
C:\Users\ASD\AppData\Local\Temp\ipykernel_1132\3687761658.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()
C:\Users\ASD\AppData\Local\Temp\ipykernel_1132\3687761658.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.

✅ Dataset ready: 169888 samples, 377 symptoms, 678 unique diseases.


In [12]:
# ================================================
# ⚖️ Class Weights (based on rarity) — FIXED
# ================================================

# Average "Weight" per disease (rarity info)
disease_weight_map = df.groupby("diseases")["Weight"].mean().to_dict()

# Frequency of each disease (number of times it appears in multi-label matrix)
class_counts = Y.sum(axis=0)
class_freq = class_counts / np.sum(class_counts)

# Compute inverse frequency weights
inv_freq = 1.0 / (class_freq + 1e-6)  # add epsilon to avoid div by zero
inv_freq = inv_freq / np.max(inv_freq)  # normalize to range [0, 1]

# Combine with rarity weights (lower weight for very common diseases)
final_weights = {}
for i, disease in enumerate(mlb.classes_):
    rarity_weight = 1 / disease_weight_map.get(disease, 2)  # smaller for common
    final_weights[i] = float(inv_freq[i] * rarity_weight)

print("✅ Computed class weights for multi-label setup.")
print(list(final_weights.items())[:10])


✅ Computed class weights for multi-label setup.
[(0, 0.002028606954810812), (1, 0.014800259191213335), (2, 0.002242987371067453), (3, 0.003934244592456649), (4, 0.11461636793654426), (5, 0.0036227439303506814), (6, 0.06472695212888784), (7, 0.05837471689365878), (8, 0.023613562626594732), (9, 0.003885285025684442)]


In [18]:
model = Sequential([
    Dense(512, input_dim=input_dim, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='sigmoid')  # sigmoid for multi-label
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

C:\Users\ASD\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:

early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [13]:

X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

In [15]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    class_weight=final_weights,
    callbacks=[early_stop],
    verbose=2
)


Epoch 1/50
2124/2124 - 50s - 23ms/step - accuracy: 0.1033 - loss: 4.5381e-05 - val_accuracy: 0.4381 - val_loss: 0.0060
Epoch 2/50
2124/2124 - 63s - 29ms/step - accuracy: 0.4353 - loss: 1.2081e-05 - val_accuracy: 0.7630 - val_loss: 0.0032
Epoch 3/50
2124/2124 - 41s - 19ms/step - accuracy: 0.6436 - loss: 7.7523e-06 - val_accuracy: 0.8399 - val_loss: 0.0022
Epoch 4/50
2124/2124 - 39s - 18ms/step - accuracy: 0.7269 - loss: 6.0728e-06 - val_accuracy: 0.8626 - val_loss: 0.0018
Epoch 5/50
2124/2124 - 39s - 18ms/step - accuracy: 0.7661 - loss: 5.1953e-06 - val_accuracy: 0.8734 - val_loss: 0.0017
Epoch 6/50
2124/2124 - 38s - 18ms/step - accuracy: 0.7873 - loss: 4.6828e-06 - val_accuracy: 0.8813 - val_loss: 0.0015
Epoch 7/50
2124/2124 - 38s - 18ms/step - accuracy: 0.8028 - loss: 4.2964e-06 - val_accuracy: 0.8808 - val_loss: 0.0015
Epoch 8/50
2124/2124 - 39s - 18ms/step - accuracy: 0.8135 - loss: 4.0724e-06 - val_accuracy: 0.8846 - val_loss: 0.0014
Epoch 9/50
2124/2124 - 40s - 19ms/step - accurac

In [16]:
# 💾 Save Model and Mappings
# ================================================
os.makedirs("saved_model", exist_ok=True)
model.save("saved_model/disease_predictor_multilabel.h5")

with open("saved_model/mlb.pkl", "wb") as f:
    pickle.dump(mlb, f)
with open("saved_model/symptom_mapping.pkl", "wb") as f:
    pickle.dump(symptom_cols, f)

print("\n✅ Model and encoders saved successfully!")


✅ Model and encoders saved successfully!


In [ ]:
# 🔮 Prediction Function (Top-N)
# ================================================
def predict_disease(symptom_list, top_n=5):
    # Load saved model & mappings
    mdl = tf.keras.models.load_model("saved_model/disease_predictor_multilabel.h5")
    with open("saved_model/mlb.pkl", "rb") as f:
        mlb = pickle.load(f)
    with open("saved_model/symptom_mapping.pkl", "rb") as f:
        symptom_cols = pickle.load(f)

    # Create binary input vector
    symptom_list = [s.strip().lower() for s in symptom_list]
    input_vector = np.array([[1 if col.lower() in symptom_list else 0 for col in symptom_cols]])

    # Predict probabilities
    probs = mdl.predict(input_vector)[0]
    top_indices = probs.argsort()[-top_n:][::-1]
    top_diseases = [(mlb.classes_[i], round(float(probs[i] * 100), 2)) for i in top_indices]

    # Normalize for confidence scaling
    total = sum([p for _, p in top_diseases])
    normalized = [(d, round(p / total * 100, 2)) for d, p in top_diseases]

    return normalized

In [ ]:
# 🧪 Example Test
# ================================================
test_symptoms = ["fever", "cough", "chills"]
preds = predict_disease(test_symptoms, top_n=5)
print("\n🩺 Top Predictions:")
for d, conf in preds:
    print(f"• {d} — {conf}%")
print(preds)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 296ms/step
[{'disease': 'white blood cell disease', 'confidence': 37.18}, {'disease': 'acute respiratory distress syndrome (ards)', 'confidence': 17.5}, {'disease': 'common cold', 'confidence': 16.5}, {'disease': 'pneumonia', 'confidence': 14.81}, {'disease': 'flu', 'confidence': 14.02}]

🩺 Top Predictions:
• disease — confidence%
• disease — confidence%
• disease — confidence%
• disease — confidence%
• disease — confidence%


In [8]:
test_cases = [
    (["fever", "cough", "fatigue"], ["common cold", "flu", "viral infection"]),
    (["chest pain", "shortness of breath"], ["heart attack", "angina", "pneumonia"]),
    (["headache", "nausea", "vomiting"], ["migraine", "food poisoning", "heat exhaustion"]),
    (["itching", "skin rash"], ["allergy", "drug reaction"]),
    (["joint pain", "fatigue", "fever"], ["arthritis", "dengue"]),
    (["sore throat", "fever", "cough"], ["tonsillitis", "flu", "bronchitis"]),
    (["abdominal pain", "vomiting", "diarrhea"], ["gastroenteritis", "food poisoning"]),
    (["anxiety", "restlessness", "palpitations"], ["panic disorder", "anxiety disorder"]),
    (["runny nose", "sneezing"], ["allergic rhinitis", "common cold"]),
    (["high fever", "rash", "joint pain"], ["dengue", "chikungunya"]),
    (["breathlessness", "cough", "wheezing"], ["asthma", "bronchitis"]),
    (["fatigue", "weight loss", "night sweats"], ["tuberculosis", "cancer"])
]

In [17]:
# Run tests
passed = 0
failed = 0
results = []

for i, (symptoms, expected) in enumerate(test_cases, start=1):
    preds = predict_disease(symptoms, top_n=3)
    pred_diseases = [d for d, _ in preds]
    match = any(e.lower() in [p.lower() for p in pred_diseases] for e in expected)

    status = "✅ PASS" if match else "❌ FAIL"
    if match:
        passed += 1
    else:
        failed += 1

    results.append({
        "Test Case": i,
        "Input Symptoms": symptoms,
        "Predicted": pred_diseases,
        "Expected": expected,
        "Result": status
    })

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step
[]


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step
['chest pain']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 228ms/step
[]


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 265ms/step
['itching']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 220ms/step
[]


...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step
['anxiety']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step
['runny nose']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step
['high fever', 'rash']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
['breathlessness']


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step
['weight loss', 'night sweats']


In [22]:
# Display summary
print("\n==== 🧪 MODEL TEST SUITE RESULTS ====\n")
for r in results:
    print(f"Test {r['Test Case']}: {r['Result']}")
    print(f"  Symptoms: {r['Input Symptoms']}")
    print(f"  Predicted: {r['Predicted']}")
    print(f"  Expected: {r['Expected']}\n")

print(f"✅ PASSED: {passed} / {len(test_cases)}")
print(f"❌ FAILED: {failed} / {len(test_cases)}")

if failed == 0:
    print("\n🎯 All test cases passed successfully!")
else:
    print("\n⚠️ Some test cases failed — consider re-training or tuning the model.")


==== 🧪 MODEL TEST SUITE RESULTS ====

Test 1: ❌ FAIL
  Symptoms: ['fever', 'cough', 'fatigue']
  Predicted: ['white blood cell disease', 'muscular dystrophy', 'hemolytic anemia']
  Expected: ['common cold', 'flu', 'viral infection']

Test 2: ❌ FAIL
  Symptoms: ['chest pain', 'shortness of breath']
  Predicted: ['cysticercosis', 'hypernatremia', 'adrenal adenoma']
  Expected: ['heart attack', 'angina', 'pneumonia']

Test 3: ❌ FAIL
  Symptoms: ['headache', 'nausea', 'vomiting']
  Predicted: ['carbon monoxide poisoning', 'pain after an operation', 'ileus']
  Expected: ['migraine', 'food poisoning', 'heat exhaustion']

Test 4: ❌ FAIL
  Symptoms: ['itching', 'skin rash']
  Predicted: ['acariasis', 'scabies', 'dyshidrosis']
  Expected: ['allergy', 'drug reaction']

Test 5: ❌ FAIL
  Symptoms: ['joint pain', 'fatigue', 'fever']
  Predicted: ['croup', 'primary insomnia', 'Other']
  Expected: ['arthritis', 'dengue']

Test 6: ❌ FAIL
  Symptoms: ['sore throat', 'fever', 'cough']
  Predicted: ['ly

In [15]:
from tensorflow.keras.models import load_model

# Load trained model
model = load_model("saved_model/disease_predictor_multilabel.h5")

# Load the MultiLabelBinarizer and symptom mapping
with open("saved_model/mlb.pkl", "rb") as f:
    mlb = pickle.load(f)
with open("saved_model/symptom_mapping.pkl", "rb") as f:
    symptom_cols = pickle.load(f)


In [17]:
loss, accuracy = model.evaluate(X_val, y_val, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")


Test Loss: 0.0010
Test Accuracy: 0.9010


In [20]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 512)            │       193,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 678)            │        87,462 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 448,294 (1.71 MB)

 Trainable params: 446,758 (1.70 MB)

 Non-trainable params: 1,536 (6.00 KB)